# Step 2:- Data Transformation or Chunking

In [3]:
# Path helps us work with files and folders in a platform-independent way.
from pathlib import Path

# PyPDFLoader extracts text from PDF files and creates LangChain Document objects.
from langchain_community.document_loaders import PyPDFLoader

# RecursiveCharacterTextSplitter divides large documents into smaller chunks.
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
from pathlib import Path

# Folder containing all PDFs used as our knowledge base.
documents_path = Path("data/documents")

# Find all PDF files in the folder.
pdf_files = list(documents_path.glob("*.pdf"))

print("PDF files found:", len(pdf_files))

# Display the filenames we found.
for pdf in pdf_files:
    print("-", pdf.name)

PDF files found: 3
- Attention Is All You Need.pdf
- BERT Pre-training of Deep Bidirectional Transformers for.pdf
- LLM.pdf


In [7]:
# This list will store every page from every PDF.
all_documents = []

# Load each PDF one by one.
for pdf_file in pdf_files:

    # Create a PDF loader for the current file.
    loader = PyPDFLoader(str(pdf_file))

    # Load all pages from the PDF.
    docs = loader.load()

    # Add the PDF filename to each page's metadata.
    # This will be useful later for source citations.
    for doc in docs:
        doc.metadata["source_file"] = pdf_file.name

    # Add the pages to our complete document collection.
    all_documents.extend(docs)

print("Total pages loaded:", len(all_documents))

Total pages loaded: 77


In [8]:
# Check the type of our complete collection.
print("Type of all_documents:", type(all_documents))

# Check the type of a single page/document.
print("Type of first document:", type(all_documents[0]))

# Display a small portion of the first page's text.
print("\nFirst page content:\n")
print(all_documents[0].page_content[:500])

# Display the metadata attached to that page.
print("\nMetadata:\n")
print(all_documents[0].metadata)

Type of all_documents: <class 'list'>
Type of first document: <class 'langchain_core.documents.base.Document'>

First page content:

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K

Metadata:

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data\

In [9]:
# RecursiveCharacterTextSplitter divides large documents
# into smaller chunks that can later be embedded and retrieved.

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initial configuration.
# We will experiment with these values later.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

print("Text splitter created successfully.")
print("Chunk size:", text_splitter._chunk_size)
print("Chunk overlap:", text_splitter._chunk_overlap)

Text splitter created successfully.
Chunk size: 1000
Chunk overlap: 200


In [10]:
# Split every Document/page into smaller Document chunks.
chunks = text_splitter.split_documents(all_documents)

print("Original documents/pages:", len(all_documents))
print("Total chunks created:", len(chunks))

Original documents/pages: 77
Total chunks created: 481


In [11]:
# Inspect the first chunk.
print("First chunk:")
print(chunks[0].page_content)

# Check the metadata preserved from the original document.
print("\nMetadata:")
print(chunks[0].metadata)

# Check the size of this chunk.
print("\nChunk length:", len(chunks[0].page_content))

First chunk:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions

M

In [12]:
# We will test different chunk sizes to see how they
# affect the total number of chunks.

chunk_sizes = [500, 1000, 1500]

for size in chunk_sizes:
    # Create a splitter with the current chunk size.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=200
    )

    # Split all our documents using this configuration.
    test_chunks = splitter.split_documents(all_documents)

    print(f"Chunk size = {size} → {len(test_chunks)} chunks")

Chunk size = 500 → 1239 chunks
Chunk size = 1000 → 481 chunks
Chunk size = 1500 → 312 chunks


In [13]:
# We keep the chunk size fixed at 1000
# and change only the overlap.

overlap_values = [0, 100, 200, 300]

for overlap in overlap_values:

    # Create a splitter with the current overlap.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=overlap
    )

    # Create chunks using this configuration.
    test_chunks = splitter.split_documents(all_documents)

    print(
        f"Chunk overlap = {overlap:3} → "
        f"{len(test_chunks)} chunks"
    )

Chunk overlap =   0 → 416 chunks
Chunk overlap = 100 → 443 chunks
Chunk overlap = 200 → 481 chunks
Chunk overlap = 300 → 540 chunks


In [14]:
# Our initial chunking configuration for VeriRAG.
# We will later evaluate whether these values give good retrieval quality.

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

# Create the final chunks that will be passed to the embedding stage.
chunks = text_splitter.split_documents(all_documents)

print("Final chunking configuration")
print("-----------------------------")
print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)
print("Total chunks:", len(chunks))

Final chunking configuration
-----------------------------
Chunk size: 1000
Chunk overlap: 200
Total chunks: 481
